<a href="https://colab.research.google.com/github/anirbanghoshsbi/create_knowledge_base/blob/main/Do_hdbscan_clustering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# OPTION 1:
# If files are directly in a GitHub repo,
# easiest approach is git clone.

!git clone https://github.com/anirbanghoshsbi/create_knowledge_base.git


Cloning into 'create_knowledge_base'...
remote: Enumerating objects: 95, done.
remote: Counting objects: 100% (95/95), done.
remote: Compressing objects: 100% (93/93), done.
remote: Total 95 (delta 43), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (95/95), 81.44 KiB | 11.63 MiB/s, done.
Resolving deltas: 100% (43/43), done.


In [8]:
# =========================================================
# INSTALL
# =========================================================

#!pip install -q hdbscan scikit-learn matplotlib pandas


# =========================================================
# IMPORTS
# =========================================================

import json
import numpy as np
import pandas as pd
import hdbscan

from collections import defaultdict
from sklearn.metrics.pairwise import cosine_similarity


# =========================================================
# LOAD EMBEDDINGS
# =========================================================

# Path from previous step
EMBEDDING_FILE = "kb_output/concept_embeddings.json"

with open(EMBEDDING_FILE, "r", encoding="utf-8") as f:
    embedding_data = json.load(f)

print(f"Loaded {len(embedding_data)} embeddings")


# =========================================================
# CONVERT TO MATRIX
# =========================================================

concept_ids = []
embedding_vectors = []

for item in embedding_data:

    concept_ids.append(item["concept_id"])

    embedding_vectors.append(item["embedding"])

embedding_vectors = np.array(embedding_vectors)

print("Embedding matrix shape:")
print(embedding_vectors.shape)


# =========================================================
# RUN HDBSCAN CLUSTERING
# =========================================================

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=2,          # minimum concepts per cluster
    min_samples=1,
    metric='euclidean',
    cluster_selection_method='eom'
)

cluster_labels = clusterer.fit_predict(embedding_vectors)

print("\nClustering complete")


# =========================================================
# LOAD MASTER CONCEPTS
# =========================================================

MASTER_JSON = "kb_output/master_concepts.json"

with open(MASTER_JSON, "r", encoding="utf-8") as f:
    master_concepts = json.load(f)

# Fast lookup
concept_lookup = {
    c["concept_id"]: c
    for c in master_concepts
}


# =========================================================
# GROUP CONCEPTS BY CLUSTER
# =========================================================

clusters = defaultdict(list)

for concept_id, label in zip(concept_ids, cluster_labels):

    clusters[label].append(concept_id)


# =========================================================
# PRINT CLUSTERS
# =========================================================

print("\n" + "=" * 80)
print("DISCOVERED CLUSTERS")
print("=" * 80)

for cluster_id, concept_group in clusters.items():

    # HDBSCAN uses -1 for noise/outliers
    if cluster_id == -1:
        continue

    print("\n")
    print("=" * 60)
    print(f"CLUSTER {cluster_id}")
    print("=" * 60)

    for cid in concept_group:

        concept = concept_lookup[cid]

        print(f"\n[{cid}]")
        print(f"Title: {concept['title']}")
        print(f"Type: {concept['type']}")
        print(f"Statement: {concept['concise_statement']}")


# =========================================================
# SAVE CLUSTER RESULTS
# =========================================================

cluster_output = []

for concept_id, label in zip(concept_ids, cluster_labels):

    concept = concept_lookup[concept_id]

    cluster_output.append({

        "concept_id": concept_id,

        "cluster_id": int(label),

        "title": concept["title"],

        "type": concept["type"],

        "statement": concept["concise_statement"]

    })


CLUSTER_SAVE_PATH = "kb_output/clustered_concepts.json"

with open(CLUSTER_SAVE_PATH, "w", encoding="utf-8") as f:
    json.dump(cluster_output, f, indent=2, ensure_ascii=False)

print("\n")
print("=" * 80)
print(f"Cluster results saved to:\n{CLUSTER_SAVE_PATH}")
print("=" * 80)


# =========================================================
# OPTIONAL — PRINT NOISE CONCEPTS
# =========================================================

print("\n")
print("=" * 80)
print("NOISE / OUTLIER CONCEPTS")
print("=" * 80)

noise_count = 0

for concept_id, label in zip(concept_ids, cluster_labels):

    if label == -1:

        concept = concept_lookup[concept_id]

        noise_count += 1

        print(f"\n[{concept_id}]")
        print(concept["title"])

print(f"\nTotal noise concepts: {noise_count}")


# =========================================================
# OPTIONAL — CLUSTER SUMMARY
# =========================================================

print("\n")
print("=" * 80)
print("CLUSTER SUMMARY")
print("=" * 80)

valid_clusters = [
    cid for cid in clusters.keys()
    if cid != -1
]

print(f"Total clusters found: {len(valid_clusters)}")

for cid in valid_clusters:

    print(
        f"Cluster {cid}: "
        f"{len(clusters[cid])} concepts"
    )

Loaded 189 embeddings
Embedding matrix shape:
(189, 768)

Clustering complete

DISCOVERED CLUSTERS


CLUSTER 39

[PM_00001]
Title: History Is Path Dependent
Type: mental model
Statement: Small random events can create massive long-term consequences through cascading effects.

[PM_00011]
Title: History Is Path-Dependent
Type: causal_relationship
Statement: Major outcomes often emerge from tiny, random initial conditions that compound over time.

[PM_00018]
Title: Complex Systems Have Infinite Causal Chains
Type: mental_model
Statement: Every major event emerges from deep interconnected causal chains extending far into the past.


CLUSTER 31

[PM_00002]
Title: The World Is Fragile
Type: risk concept
Statement: Many important systems appear stable until a hidden vulnerability suddenly breaks them.

[PM_00039]
Title: Visible Stability Often Masks Hidden Fragility
Type: risk_concept
Statement: Systems that appear stable may remain highly vulnerable to unseen shocks.


CLUSTER 25

[PM_00003]